# ChromaDB Inspector

Query and debug ChromaDB collections for each chunking strategy.

**Collections:**
- `squad_baseline_258_tok` → `./squad_chroma_db_baseline_258_tok`
- `squad_most_recent_low_acc_258t_w128_wtd` → `./squad_chroma_db_most_recent_low_acc_258t_w128_wtd`
- `squad_most_recent_258t_w254` → `./squad_chroma_db_most_recent_258t_w254`
- `squad_nonlinear_258t_w254` → `./squad_chroma_db_nonlinear_258t_w254`

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve().parents[0]
sys.path.append(str(REPO_ROOT))

import chromadb
from sentence_transformers import SentenceTransformer
from index_documents import LocalEmbeddingFunction
from config import EMBEDDING_MODEL_NAME

STRATEGIES = [
    'baseline_258_tok',
    'most_recent_low_acc_258t_w128_wtd',
    'most_recent_258t_w254',
    'nonlinear_258t_w254',
]

print(f'Embedding model: {EMBEDDING_MODEL_NAME}')
model = SentenceTransformer(EMBEDDING_MODEL_NAME)
embed_fn = LocalEmbeddingFunction(model)
print('Model loaded.')

In [ ]:
def get_collection(strategy: str) -> chromadb.Collection:
    client = chromadb.PersistentClient(
        path=str(REPO_ROOT / f'squad_chroma_db_{strategy}')
    )
    return client.get_collection(
        name=f'squad_{strategy}',
        embedding_function=embed_fn,
    )

## Collection Sizes

In [ ]:
for strategy in STRATEGIES:
    try:
        col = get_collection(strategy)
        print(f'{strategy:45s} {col.count():>6} chunks')
    except Exception as e:
        print(f'{strategy:45s} ERROR: {e}')

## Peek at Records

Inspect the first N records in a collection — useful for verifying source name format.

In [ ]:
STRATEGY = 'baseline_258_tok'  # change to inspect a different strategy
N        = 10

col = get_collection(STRATEGY)
results = col.get(limit=N, include=['metadatas', 'documents'])

for meta, doc in zip(results['metadatas'], results['documents']):
    print(f"{meta}")
    print(f"  {repr(doc[:80])}")
    print()

## All Unique Source Names

Lists every distinct `source` value stored in a collection.
Compare against what `evaluate_squad.py` constructs to diagnose "article not found" mismatches.

In [ ]:
STRATEGY = 'baseline_258_tok'  # change to inspect a different strategy

col = get_collection(STRATEGY)
all_meta = col.get(include=['metadatas'])
sources = sorted(set(m['source'] for m in all_meta['metadatas']))

print(f'{len(sources)} unique sources in {STRATEGY}:')
for s in sources[:30]:
    print(f'  {s}')
if len(sources) > 30:
    print(f'  ... ({len(sources) - 30} more)')

## Look Up a Specific Article

Check whether a particular article exists in a collection and inspect its chunks.

In [ ]:
STRATEGY     = 'baseline_258_tok'
ARTICLE_NAME = '2008 Sichuan earthquake.txt'  # must match source format exactly

col = get_collection(STRATEGY)
results = col.get(
    where={'source': {'$eq': ARTICLE_NAME}},
    include=['metadatas', 'documents'],
)

print(f'Found {len(results["ids"])} chunks for "{ARTICLE_NAME}" in {STRATEGY}')
for meta, doc in zip(results['metadatas'], results['documents']):
    print(f"  chunk {meta['chunk_index']:3d}: {repr(doc[:80])}")

## Find Missing Articles Across Strategies

Compares source names across all four collections to find articles present in some but not others.

In [ ]:
import pandas as pd

strategy_sources = {}
for strategy in STRATEGIES:
    try:
        col = get_collection(strategy)
        all_meta = col.get(include=['metadatas'])
        strategy_sources[strategy] = set(m['source'] for m in all_meta['metadatas'])
    except Exception as e:
        print(f'Could not load {strategy}: {e}')
        strategy_sources[strategy] = set()

all_sources = set.union(*strategy_sources.values())
rows = []
for source in sorted(all_sources):
    row = {'source': source}
    for strategy in STRATEGIES:
        row[strategy[:12]] = '✓' if source in strategy_sources[strategy] else '✗'
    rows.append(row)

df = pd.DataFrame(rows)
missing = df[(df.iloc[:, 1:] == '✗').any(axis=1)]
print(f'{len(missing)} articles missing from at least one strategy:')
display(missing.reset_index(drop=True))

## Test Retrieval

Run a retrieval query against a collection to see what chunks come back — the same call `evaluate_squad.py` makes.

In [ ]:
STRATEGY     = 'baseline_258_tok'
QUESTION     = 'What fault did the 2008 Sichuan earthquake occur on?'
ARTICLE_NAME = '2008 Sichuan earthquake.txt'  # set to None for global retrieval
N_RESULTS    = 10

col = get_collection(STRATEGY)
kwargs = dict(
    query_texts=[QUESTION],
    n_results=N_RESULTS,
    include=['documents', 'metadatas', 'distances'],
)
if ARTICLE_NAME:
    kwargs['where'] = {'source': {'$in': [ARTICLE_NAME]}}

results = col.query(**kwargs)

print(f'Query: {QUESTION!r}')
print(f'Filter: {ARTICLE_NAME or "none (global)"}')
print()
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0],
)):
    print(f'[{i+1}] chunk={meta["chunk_index"]}  distance={dist:.4f}')
    print(f'     {repr(doc[:120])}')
    print()